# вернуть маятник в заданное положение

Начальный угол известен, начальную скорость $s=\dot\theta(0)$ подбираем методом стрельбы.
Для каждого $s$ решаем задачу Коши методом RK4 с постоянным шагом и вычисляем

$$R(s)=\theta(t_f;s)-\theta_f.$$

Сначала рассматриваем линейный маятник, затем заменяем $\theta$ в уравнении на $\sin\theta$.
Углы измеряются в радианах, время безразмерное.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from matplotlib.ticker import NullLocator
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path
from tempfile import TemporaryDirectory

In [ ]:
def rhs(y, nonlinear):  # правая часть системы для одного или нескольких маятников
    force = np.sin(y[..., 0]) if nonlinear else y[..., 0]
    return np.stack((y[..., 1], -force), axis=-1)

def rk4_step(y, h, nonlinear):  # один шаг метода Рунге — Кутты четвёртого порядка
    k1 = rhs(y, nonlinear)
    k2 = rhs(y + h * k1 / 2, nonlinear)
    k3 = rhs(y + h * k2 / 2, nonlinear)
    k4 = rhs(y + h * k3, nonlinear)
    return y + h * (k1 + 2*k2 + 2*k3 + k4) / 6

def integrate(s, t_end, h, nonlinear=False):  # траектория с начальным углом 0 и скоростью s
    s = np.asarray(s, dtype=float)
    n = round(t_end / h)
    if n < 1 or not np.isclose(n*h, t_end):
        raise ValueError('время расчёта должно быть кратно положительному шагу')
    t = np.linspace(0, t_end, n + 1)
    y = np.empty((n + 1,) + s.shape + (2,))
    y[0] = np.stack((np.zeros_like(s), s), axis=-1)
    for i in range(n):
        y[i + 1] = rk4_step(y[i], h, nonlinear)
    return t, y

def make_residual(t_end, target, h, nonlinear=False):  # невязка и счётчик её вычислений
    def residual(s):
        residual.calls += np.size(s)
        _, y = integrate(s, t_end, h, nonlinear)
        return y[-1, ..., 0] - target
    residual.calls = 0
    return residual

## подбор начальной скорости

В методе Ньютона используем

$$s_{k+1}=s_k-\frac{R(s_k)}{R'(s_k)},\qquad
R'(s)\approx\frac{R(s+\varepsilon)-R(s-\varepsilon)}{2\varepsilon},
\qquad \varepsilon=10^{-5}.$$

Остановка по невязке: $|R(s)|\leq\text{tol}$. Два дополнительных вычисления для производной входят в счётчик.
В истории сохраняем скорость, невязку и число её вычислений. При решении методом бисекции заранее проверяем смену знака на концах интервала.

In [ ]:
def newton(f, s0, tol=1e-10, ds=1e-5, max_iter=40, derivative_tol=1e-9):  # метод Ньютона с центральной разностью для производной
    s = float(s0)
    history = []
    for k in range(max_iter + 1):
        r = float(f(s))
        history.append((s, r, f.calls))
        if not np.isfinite(r):
            return s, 'нечисловое значение', np.array(history)
        if abs(r) <= tol:
            return s, 'сошёлся', np.array(history)
        if k == max_iter:
            return s, 'лимит итераций', np.array(history)
        derivative = (float(f(s + ds)) - float(f(s - ds))) / (2*ds)
        if not np.isfinite(derivative) or abs(derivative) < derivative_tol:
            return s, 'малая производная', np.array(history)
        next_s = s - r / derivative
        if not np.isfinite(next_s):
            return s, 'нечисловое значение', np.array(history)
        if next_s == s:
            return s, 'остановка шага', np.array(history)
        s = next_s

def bisection(f, a, b, tol=1e-10, max_iter=100):  # бисекция с сохранением уже вычисленных значений на концах
    fa = float(f(a)); history = [(a, fa, f.calls)]
    fb = float(f(b)); history.append((b, fb, f.calls))
    if abs(fa) <= tol:
        return a, 'сошёлся', np.array(history)
    if abs(fb) <= tol:
        return b, 'сошёлся', np.array(history)
    if fa * fb > 0:
        raise ValueError('на концах интервала нужны разные знаки невязки')
    for k in range(max_iter):
        c = (a + b) / 2
        fc = float(f(c)); history.append((c, fc, f.calls))
        if abs(fc) <= tol:
            return c, 'сошёлся', np.array(history)
        if c == a or c == b:
            return c, 'остановка шага', np.array(history)
        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc
    return c, 'лимит итераций', np.array(history)

## линейный маятник

$$\ddot\theta+\theta=0,\qquad \theta(0)=0,\qquad \theta(2)=0.2.$$

Точное решение и начальная скорость:

$$\theta(t)=\frac{0.2\sin t}{\sin 2},\qquad s_* = \frac{0.2}{\sin 2}.$$

Для первого расчёта берём $h=0.1$. Для метода Ньютона выбираем начальное приближение $s_0=0.8$, для метода бисекции — интервал $[0,1]$.

In [ ]:
h_linear = 0.1
s_exact = 0.2 / np.sin(2)
f_newton = make_residual(2, 0.2, h_linear)
f_bisect = make_residual(2, 0.2, h_linear)
s_newton, status_newton, history_newton = newton(f_newton, 0.8)
s_bisect, status_bisect, history_bisect = bisection(f_bisect, 0, 1)
print(f'Точная скорость: {s_exact:.12f}')
for name, s, status, f in [('Ньютон', s_newton, status_newton, f_newton),
                           ('Бисекция', s_bisect, status_bisect, f_bisect)]:
    print(f'{name:9s}: s = {s:.12f}, {status}, вычислений невязки: {f.calls}')
print(f'Невязки на концах: {history_bisect[0, 1]:.4f}, {history_bisect[1, 1]:.4f}')

trials = [-0.2, 0, 0.15, 0.3, 0.5]
t, y = integrate(trials, 2, h_linear)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for j, s in enumerate(trials):
    ax.plot(t, y[:, j, 0], label=f's = {s:g}')
ax.plot(2, 0.2, '*', color='black', markersize=12, label='заданное положение')
ax.set(xlabel='t', ylabel='θ', title='пристрелочные траектории')
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
plt.show()

In [ ]:
s_grid = np.linspace(-0.1, 1, 200)
f_plot = make_residual(2, 0.2, h_linear)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(s_grid, f_plot(s_grid), color='gray', label='R(s)')
axes[0].axhline(0, color='black', linewidth=0.7)
for history, name, color, marker in [(history_newton, 'Ньютон', 'tab:blue', 'o'),
                                    (history_bisect, 'бисекция', 'tab:orange', 'x')]:
    axes[0].plot(history[:, 0], history[:, 1], marker, color=color,
                 markersize=5, label=name)
    axes[1].semilogy(history[:, 2], np.abs(history[:, 1]), marker+'-',
                     color=color, markersize=4, label=name)
axes[0].set(xlabel='s', ylabel='R(s)', title='невязка и приближения')
axes[1].set(xlabel='число вычислений невязки', ylabel='|R|', title='сходимость методов')
axes[1].axhline(1e-10, color='gray', linestyle=':', label='допуск')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
plt.show()

## шаг интегрирования и точность решения

При каждом новом $h$ заново подбираем $s$ с допуском $10^{-13}$.
Сравниваем движение с точным решением на узлах сетки.
Отдельно вычисляем $s\sin 2-0.2$: это невязка найденной скорости для точного уравнения, без ошибки метода RK4.

In [ ]:
steps = np.array([0.2, 0.1, 0.05, 0.025])
linear_results = []
print('    h          s             |s − s*|      max |θ − θ*|    |s sin 2 − 0.2|')
for h in steps:
    f = make_residual(2, 0.2, h)
    s, status, _ = newton(f, 0.8, tol=1e-13)
    t, y = integrate(s, 2, h)
    error = y[:, 0] - s_exact * np.sin(t)
    linear_results.append((t, y, s, error))
    print(f'{h:6.3f}  {s:.12f}    {abs(s-s_exact):.3e}      {np.max(abs(error)):.3e}       {abs(s*np.sin(2)-0.2):.3e}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
t_dense = np.linspace(0, 2, 401)
axes[0].plot(t_dense, s_exact*np.sin(t_dense), color='black', label='точное решение')
t, y, _, _ = linear_results[0]
axes[0].plot(t, y[:, 0], 'o', fillstyle='none', color='tab:blue', label='RK4, h = 0.2')
for h, (t, y, s, error) in zip(steps, linear_results):
    axes[1].plot(t, error, label=f'h = {h:g}')
max_errors = np.array([np.max(abs(row[3])) for row in linear_results])
axes[2].loglog(steps, max_errors, 'o-', label='максимальная ошибка')
axes[2].loglog(steps, max_errors[0]*(steps/steps[0])**4, '--', color='gray', label='пропорционально h⁴')
axes[0].set(xlabel='t', ylabel='θ')
axes[1].set(xlabel='t', ylabel='θ − θ*')
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[2].set(xlabel='h', ylabel='max |θ − θ*|')
axes[2].set_xticks(steps, labels=[f'{h:g}' for h in steps])
axes[2].xaxis.set_minor_locator(NullLocator())
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
f = make_residual(2, 0.2, h_linear)
s_discrete, _, _ = newton(f, 0.8, tol=1e-13)
print('h = 0.1; s_h — корень численной невязки с допуском 1e-13')
print(' допуск       |R(s)|        |s − s_h|      max |θ − θ*|')
for tol in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    f = make_residual(2, 0.2, h_linear)
    s, _, hist = bisection(f, 0, 1, tol=tol)
    t, y = integrate(s, 2, h_linear)
    error = np.max(abs(y[:, 0] - s_exact*np.sin(t)))
    print(f'{tol:8.0e}    {abs(hist[-1, 1]):.3e}      {abs(s-s_discrete):.3e}      {error:.3e}')

При $h=0.1$ решение методом Ньютона потребовало 4 вычисления невязки, методом бисекции — 30 при одном допуске $10^{-10}$.
Для линейной задачи невязка линейна по $s$, поэтому достаточно одного шага метода Ньютона; это не общее свойство нелинейных задач.

Уменьшение допуска уточняет корень **численной** невязки. Само движение при этом всё ещё вычисляется с шагом $h$.
Поэтому при $h=0.1$ ошибка траектории остаётся около $2.3\cdot10^{-7}$ даже при очень малом допуске поиска корня.
После уменьшения шага до $0.025$ она падает примерно до $9.3\cdot10^{-10}$.
Невязка, ошибка подбора скорости и погрешность интегрирования — разные величины.

## нелинейный маятник

$$\ddot\theta+\sin\theta=0,\qquad \theta(0)=\theta(4)=0,
\qquad -1.99\leq s\leq1.99.$$

Невязка равна $R(s)=\theta(4;s)$. Угол не приводим по модулю $2\pi$.
На сетке скоростей находим интервалы смены знака и уточняем корни бисекцией. Нулевое решение учитываем отдельно.

In [ ]:
def find_roots(h, points=801):  # поиск смен знака невязки и уточнение корней бисекцией
    grid = np.linspace(-1.99, 1.99, points)
    grid[np.abs(grid) < 1e-14] = 0.0
    f = make_residual(4, 0, h, True)
    values = f(grid)
    roots = list(grid[np.abs(values) < 1e-13])
    for i in np.flatnonzero(values[:-1] * values[1:] < 0):
        root, status, _ = bisection(f, grid[i], grid[i+1], tol=1e-12)
        if status != 'сошёлся': raise RuntimeError(status)
        roots.append(root)
    return np.array(sorted(roots)), grid, values

In [ ]:
h_nonlinear = 0.01
roots, s_grid, residual_grid = find_roots(h_nonlinear)
colors = ['tab:blue', 'gray', 'tab:orange']
t_nl, y_nl = integrate(roots, 4, h_nonlinear, True)
print('      s              θ(4)             θ̇(4)')
for s, end in zip(roots, y_nl[-1]):
    print(f'{s: .12f}    {end[0]: .3e}     {end[1]: .12f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(s_grid, residual_grid, color='black', linewidth=1)
axes[0].axhline(0, color='gray', linewidth=0.7)
for j, (s, color) in enumerate(zip(roots, colors)):
    label = f's = {s:.6f}'
    axes[0].plot(s, y_nl[-1, j, 0], 'o', color=color, label=label)
    axes[1].plot(t_nl, y_nl[:, j, 0], color=color, label=label)
axes[0].set(xlabel='s', ylabel='R(s)', title='корни невязки')
axes[1].set(xlabel='t', ylabel='θ', title='движения с одинаковыми граничными углами')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
plt.show()

В исследованном диапазоне найдены три скорости: $s=0$ и $s\approx\pm1.604813$.
При $s=0$ маятник остаётся в покое. Два ненулевых решения симметричны: маятник отклоняется в одну из сторон и впервые возвращается в нижнее положение при $t=4$.

## зависимость сходимости метода Ньютона от начального приближения

Берём 801 начальное приближение на $[-1.99,1.99]$. Промежуточные шаги метода Ньютона не ограничиваем этим интервалом.
Для наглядности добавляем две точки экстремума численной невязки: там производная почти нулевая, поэтому деление на неё делает шаг ненадёжным.

Используем $h=0.01$, допуск $10^{-10}$ и не более 40 шагов.
При $|R'|<10^{-9}$ останавливаемся с причиной «малая производная».
Для построения карты вычисляем независимые итерации одновременно для массива скоростей.

In [ ]:
def newton_map(f, starts, tol=1e-10, ds=1e-5, max_iter=40, derivative_tol=1e-9):  # те же итерации метода Ньютона сразу для массива начальных приближений
    s = np.asarray(starts).copy()
    status = np.full(s.shape, 'работает', dtype=object)
    iterations = np.zeros(s.shape, dtype=int)
    for k in range(max_iter + 1):
        indices = np.flatnonzero(status == 'работает')
        if not len(indices): break
        r = f(s[indices]); iterations[indices] = k
        finite = np.isfinite(r)
        status[indices[~finite]] = 'нечисловое значение'
        converged = finite & (np.abs(r) <= tol)
        status[indices[converged]] = 'сошёлся'
        active = finite & ~converged
        indices, r = indices[active], r[active]
        if k == max_iter:
            status[indices] = 'лимит итераций'
            break
        if not len(indices): continue
        derivative = (f(s[indices] + ds) - f(s[indices] - ds)) / (2*ds)
        good = np.isfinite(derivative) & (np.abs(derivative) >= derivative_tol)
        status[indices[~good]] = 'малая производная'
        indices, r, derivative = indices[good], r[good], derivative[good]
        next_s = s[indices] - r / derivative
        status[indices[~np.isfinite(next_s)]] = 'нечисловое значение'
        status[indices[next_s == s[indices]]] = 'остановка шага'
        s[indices] = next_s
    return s, status, iterations

In [ ]:
f_map = make_residual(4, 0, h_nonlinear, True)
ds = 1e-5
turn = brentq(lambda s: (f_map(s+ds)-f_map(s-ds))/(2*ds), 0.1, 1.8, xtol=1e-12)
starts = np.sort(np.r_[np.linspace(-1.99, 1.99, 801), -turn, turn])
ends, statuses, iterations = newton_map(f_map, starts)
nearest = np.argmin(abs(ends[:, None] - roots[None, :]), axis=1)
converged = statuses == 'сошёлся'

fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2]}, constrained_layout=True)
for j, (s, color) in enumerate(zip(roots, colors)):
    mask = converged & (nearest == j)
    axes[0].scatter(starts[mask], np.zeros(mask.sum()), color=color,
                    marker='|', s=300, label=f's = {s:.6f}')
    axes[1].scatter(starts[mask], iterations[mask], color=color, s=7)
for reason in np.unique(statuses[~converged]):
    mask = statuses == reason
    axes[0].scatter(starts[mask], np.zeros(mask.sum()), color='tab:red',
                    marker='x', s=65, zorder=5, label=reason)
    axes[1].scatter(starts[mask], iterations[mask], color='tab:red', marker='x', s=40)
axes[0].set(yticks=[], ylim=(-0.5, 0.8), title='результат метода Ньютона')
axes[0].legend(loc='upper center', ncol=4, fontsize=8)
axes[1].set(xlabel='начальное приближение s₀', ylabel='число шагов')
axes[1].grid(alpha=0.25)
plt.show()
for j, s in enumerate(roots):
    print(f'К s = {s:.6f}: {np.sum(converged & (nearest == j))} приближений')
for reason in np.unique(statuses[~converged]):
    print(f'{reason}: {np.sum(statuses == reason)}')
print(f'Экстремумы невязки: s ≈ ±{turn:.6f}')

На равномерной сетке все 801 приближение сошлись, но не обязательно к ближайшему корню.
Вблизи экстремумов невязки деление на малую производную вызывает большой шаг, поэтому итог может резко меняться при изменении $s_0$.
Два добавленных приближения в самих экстремумах остановились по малой производной.
Карта показывает результаты для выбранной сетки; между её точками поведение может меняться.

## проверка при уменьшении шага

Для каждого шага заново ищем все три корня. За численный эталон принимаем решение методом RK4 с шагом $h_{\rm ref}=0.000625$, на котором также заново подбираем скорость.
Сравниваем положительную скорость и соответствующую траекторию с эталоном в общих узлах сеток:

$$E_s=|s_h-s_{\rm ref}|,\qquad
E_\theta=\max_n|\theta_h(t_n;s_h)-\theta_{\rm ref}(t_n;s_{\rm ref})|.$$

Для отрицательной скорости ошибки такие же по симметрии; нулевое решение сохраняется точно.

In [ ]:
h_reference = 0.000625
f_reference = make_residual(4, 0, h_reference, True)
s_reference, status_reference, _ = newton(f_reference, 1.6, tol=1e-13)
if status_reference != 'сошёлся':
    raise RuntimeError(status_reference)
t_reference, y_reference = integrate(s_reference, 4, h_reference, True)
nonlinear_steps = np.array([0.04, 0.02, 0.01, 0.005])
nonlinear_results = []
print(f'Скорость на мелкой сетке: {s_reference:.12f}')
print('    h       s₋             s₊              E_s          E_θ')
for h in nonlinear_steps:
    current_roots, _, _ = find_roots(h)
    s = current_roots[-1]
    t, y = integrate(s, 4, h, True)
    stride = round(h / h_reference)
    error_s = abs(s - s_reference)
    error_theta = np.max(abs(y[:, 0] - y_reference[::stride, 0]))
    nonlinear_results.append((current_roots, error_s, error_theta))
    print(f'{h:6.3f}  {current_roots[0]: .10f}  {s: .10f}    {error_s:.3e}    {error_theta:.3e}')

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.loglog(nonlinear_steps, [r[1] for r in nonlinear_results], 'o-', label='ошибка скорости')
ax.loglog(nonlinear_steps, [r[2] for r in nonlinear_results], 's-', label='ошибка траектории')
ax.loglog(nonlinear_steps, nonlinear_results[0][1]*(nonlinear_steps/nonlinear_steps[0])**4,
           '--', color='gray', label='пропорционально h⁴')
ax.set(xlabel='h', ylabel='абсолютная ошибка')
ax.set_xticks(nonlinear_steps, labels=[f'{h:g}' for h in nonlinear_steps])
ax.xaxis.set_minor_locator(NullLocator())
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
plt.show()

Все три решения сохраняются при уточнении. Ошибки скорости и траектории относительно мелкой сетки уменьшаются примерно как $h^4$.

Одинаковые положения в начале и конце не задают движение однозначно: начальная скорость неизвестна.
Можно всё время оставаться в покое или уйти в любую из сторон и вернуться.
Для задачи Коши нужны угол и скорость в один момент времени; здесь заданы только углы в разные моменты.

## анимация движений

Используем скорости, найденные при $h=0.005$. Все маятники имеют единичную длину и общую точку подвеса.
Цвета совпадают с графиками углов. Вертикальная линия и точки справа показывают текущий момент времени.

In [ ]:
animation_roots = nonlinear_results[-1][0]
t_animation, y_animation = integrate(animation_roots, 4, 0.005, True)
angles = y_animation[:, :, 0]
frame_indices = np.arange(0, len(t_animation), 8)
fig, (ax_pendulum, ax_angle) = plt.subplots(1, 2, figsize=(10, 4.2),
                                          gridspec_kw={'width_ratios': [1, 1.6]},
                                          constrained_layout=True)
ax_pendulum.set(xlim=(-1.15, 1.15), ylim=(-1.15, 0.55), xlabel='x', ylabel='y')
ax_pendulum.set_aspect('equal')
ax_pendulum.plot(0, 0, 'ko', markersize=5, zorder=5)
rods = []
dots = []
for j, (s, color) in enumerate(zip(animation_roots, colors)):
    rod, = ax_pendulum.plot([], [], 'o-', color=color, linewidth=1.8, markersize=8, alpha=0.85)
    rods.append(rod)
    ax_angle.plot(t_animation, angles[:, j], color=color, label=f's = {s:.6f}')
    dot, = ax_angle.plot([], [], 'o', color=color, markersize=6)
    dots.append(dot)
cursor = ax_angle.axvline(0, color='black', linewidth=0.8)
clock = ax_pendulum.set_title('t = 0.00')
ax_angle.set(xlabel='t', ylabel='θ', xlim=(0, 4), ylim=(-2.1, 2.1))
ax_angle.legend(loc='upper right', fontsize=8)
for ax in (ax_pendulum, ax_angle):
    ax.grid(alpha=0.2)


def update(frame):  # положение грузов и отметки на графике в один момент времени
    i = frame_indices[frame]
    t = t_animation[i]
    for j, (rod, dot) in enumerate(zip(rods, dots)):
        theta = angles[i, j]
        rod.set_data([0, np.sin(theta)], [0, -np.cos(theta)])
        dot.set_data([t], [theta])
    cursor.set_xdata([t, t])
    clock.set_text(f't = {t:.2f}')
    return rods + dots + [cursor, clock]


animation = FuncAnimation(fig, update, frames=len(frame_indices), interval=40, blit=False)
with TemporaryDirectory() as folder:
    animation_path = Path(folder) / 'pendulum.gif'
    animation.save(animation_path, writer=PillowWriter(fps=25), dpi=90)
    gif_bytes = animation_path.read_bytes()
plt.close(fig)
display(Image(data=gif_bytes, format='gif'))